In [ ]:
#12+12 Epoch: 3000, Train Loss: 0.130, Train R²: 0.866| Val Loss: 0.464, Val R²: 0.616|Test Loss: 0.482, Test R²: 0.515
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

In [ ]:
cds_df=pd.read_csv("<PAUSING_SOURCE_ROOT>/cds_df38510.csv")
cds_df = cds_df.iloc[:,1:9]
cds_df['transcript_id'] = cds_df['transcript_id_x'].str.split('.').str[0]
exp=pd.read_csv("<PAUSING_SOURCE_ROOT>/data/sc11619genes422cell_normalized.csv")
# 将 cds_df 按照 exp 的基因列排序
sorted_cds_df = cds_df.set_index('transcript_id').reindex(exp['Unnamed: 0']).reset_index()

# 查看排序后的结果
sorted_cds_df.head()
sorted_cds_df.fillna(0, inplace=True)

merged_df=sorted_cds_df
merged_df['transcript_id'] = merged_df['transcript_id_x'].str.split('.').str[0]
merged_df.shape

In [ ]:

merged_df.columns


In [ ]:
merged_df

In [ ]:
merged_df3 = pd.read_csv('./data/24077132kdncmergedf.csv')
merged_df3['protein'] = merged_df3['protein_x'].str.split('.').str[0]
merged_df3.shape

In [ ]:


pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/human_nc1_pause.csv')

pausing.columns = ['protein_id', "High_Pause_Countsnc1", "transcript_id_x"]
merged_df2 = pd.merge(merged_df, pausing, on='transcript_id_x', how='left')
merged_df2['High_Pause_Countsnc1'].fillna(0, inplace=True)
merged_df2.columns

In [ ]:


pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/human_nc2_pause.csv')

pausing.columns = ['protein_id', "High_Pause_Countsnc2", "transcript_id_x"]
merged_df2 = pd.merge(merged_df2, pausing, on='transcript_id_x', how='left')
merged_df2['High_Pause_Countsnc2'].fillna(0, inplace=True)
merged_df2.columns

In [ ]:
# 先导入pandas，如果你还没有导入的话
import pandas as pd

# 计算 rNC1 列中非 NaN 的数量
non_na_count = merged_df2['NC3'].notna().sum()

print(f"rNC1列中非NaN的数量是: {non_na_count}")
# 先导入pandas，如果你还没有导入的话
# 计算 rNC1 列中非 NaN 的数量
non_na_count = merged_df2['NC3'].notna().sum()

print(f"NC1列中非NaN的数量是: {non_na_count}")


In [ ]:
# 统计 gene 列中不为0的数量
non_zero_count = (merged_df2['gene'] != 0).sum()

print(f"gene列中不为0的数量是: {non_zero_count}")


In [ ]:
merged_df3.head

In [ ]:
merged_df2.head

In [ ]:
# 统计 gene 列中不为0的数量
non_zero_count = (merged_df2['NC3'] != 0).sum()

print(f"gene列中不为0的数量是: {non_zero_count}")


先建立有标签的掩码

In [ ]:
# 加载 merged_df3，确保去除 .后缀
merged_df3 = pd.read_csv('./data/24077132kdncmergedf.csv')
merged_df3['transcript_id'] = merged_df3['transcript_id'].str.split('.').str[0]

# merged_df2也去掉 . 后缀（如果之前没处理过）
merged_df2['transcript_id'] = merged_df2['transcript_id'].str.split('.').str[0]

# 创建一个掩码，标记merged_df2中哪些蛋白是有标签的（在merged_df3里面）
# 创建掩码（pandas Series）
is_labeled = merged_df2['transcript_id'].isin(merged_df3['transcript_id'])

# 转成torch tensor（注意dtype是bool）
is_labeled = torch.tensor(is_labeled.values, dtype=torch.bool)

# 现在可以用torch.where
labeled_idx = torch.where(is_labeled)[0]



In [ ]:
ppi_matrix = pd.read_csv('./data/ppi_ebi_string_ppi3ensp_lr_IntAct_corummatrix4p_pbulk11619.csv')

ppi_matrix.head()
all_sequence_outputsnew = np.load('./data/all_sequence_outputsnewbulk11619.npy')


In [ ]:
ppi_matrix = sp.coo_matrix(ppi_matrix)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import random
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score

# 设定随机种子
def seed_everything(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子8：1:1
#SEED = 12#Train R²: 0.866| Val Loss: 0.438, Val R²: 0.675|Test Loss: 0.576, Test R²: 0.379
SEED = 5

seed_everything(SEED)

# 早停器
class EarlyStopping:
    def __init__(self, patience=50):
        self.patience = patience
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def step(self, loss):
        if loss < self.best_loss:
            self.best_loss = loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

# 训练函数
def train_model(model, data, train_idx, y_true, optimizer, criterion, patience=50):
    early_stopping = EarlyStopping(patience=patience)
    model.train()
    for epoch in range(10000):  # 大循环，由early stopping控制
        model.train()
        optimizer.zero_grad()
        out, _ = model(data)
        loss = criterion(out[train_idx], y_true[train_idx])
        #print(f"train_idx: {train_idx}")
        loss.backward()
        optimizer.step()

        early_stopping.step(loss.item())
        if early_stopping.early_stop:
            print(f"Early Stopping at Epoch {epoch} with Loss {loss.item():.4f}")
            break

# 验证函数
def evaluate_model(model, data, idx, y_true):
    model.eval()
    with torch.no_grad():
        out, _ = model(data)
        loss = nn.MSELoss()(out[idx], y_true[idx])
        r2 = r2_score(y_true[idx].cpu().numpy(), out[idx].cpu().numpy())
    return loss.item(), r2

# 主Self-Learning流程
def self_learning_process(data, y, model, device, initial_labeled_idx, val_idx, pool_idx, batch_size=300, max_rounds=10):

    # 初始训练
    train_idx = initial_labeled_idx.clone()
    print(f"初始训练集大小: {len(train_idx)}")
    # 每次新模型初始化前，设置一次随机种子！
    seed_everything(SEED)

    # 定义优化器
   

    optimizer = optim.Adam(model.parameters(), lr=7e-2)
    criterion = nn.MSELoss()
    #optimizer = optim.Adam(train_model.parameters(), lr=7e-2)
    # 开始训练
    train_model(model, data, train_idx, y, optimizer, criterion)


    # 伪标签循环
    for round_num in range(max_rounds):
        if len(pool_idx) == 0:
            print("没有更多未标记样本，结束Self-Learning。")
            break

        model.eval()
        with torch.no_grad():
            outputs, _ = model(data)
        
        # 从pool里选前batch_size个
        select_size = min(batch_size, len(pool_idx))
        selected_idx = pool_idx[:select_size]
        #print(selected_idx)
        # 拿当前模型预测这些节点作为伪标签
        pseudo_labels = outputs[selected_idx].detach()

        # 更新y，把伪标签赋值
        y[selected_idx] = pseudo_labels

        # 将伪标签样本加入train_idx
        train_idx = torch.cat([train_idx, selected_idx], dim=0)

        # pool里移除这些样本
        pool_idx = pool_idx[select_size:]

        print(f"第{round_num+1}轮: 添加{select_size}个伪标签样本，总训练集大小{len(train_idx)}")

        # 重新训练
        optimizer = optim.Adam(model.parameters(), lr=7e-2)
        criterion = nn.MSELoss()
        train_model(model, data, train_idx, y, optimizer, criterion)

    # 最后在验证集上测试
    val_loss, val_r2 = evaluate_model(model, data, val_idx, y)
    print(f"最终验证集Loss: {val_loss:.4f}, 验证集R²: {val_r2:.4f}")
    return model



In [ ]:
# 设置随机种子
seed_everything(SEED)

class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.regressor = nn.Linear(32, 1)
        self.regressor_activation = nn.Sequential(
            nn.ReLU()
        )
    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause
        
        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore)), dim=1)
        x = self.fc(x)
        
        # Graph convolution layer
        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)
        
        # Regressor layer
        out = self.regressor(z)
        out=self.regressor_activation(out)
        return out, z
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")


In [ ]:
# 取出两列
set1 = set(merged_df2['Unnamed: 0'])
set2 = set(merged_df3['transcript_id'])

# 计算交集
intersection = set1 & set2  # 或者 set1.intersection(set2)

print(f"交集的数量: {len(intersection)}")

# 如果想看具体交集的元素：
#print(intersection)


In [ ]:
# 先建一个 merged_df2['Unnamed: 0'] 到 index 的映射
id_to_idx = {tid: idx for idx, tid in enumerate(merged_df2['Unnamed: 0'])}

# 然后只查 intersection 中的
labeled_idx = [id_to_idx[tid] for tid in intersection]

# 转成 Tensor
labeled_idx = torch.tensor(labeled_idx, dtype=torch.long)

print(f"最终labeled蛋白数量: {len(labeled_idx)}")  # 应该是4258


In [ ]:
import random
import numpy as np
import torch

#SEED = 42  # 你自己定义的种子数，比如42或者任何数

seed_everything(SEED)  
# # Python内置随机
# random.seed(SEED)
# # numpy随机
# np.random.seed(SEED)
# # torch随机
# torch.manual_seed(SEED)
# # 如果用的是GPU，也加上下面这行
# torch.cuda.manual_seed(SEED)
# torch.cuda.manual_seed_all(SEED)  # 如果有多个GPU

# # 确保CUDA中的卷积算法确定性
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False
import random
import numpy as np
import torch

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# 只选有标签的数据
X_cpm_log2 = np.log2((merged_df2[['rNC2']].values / np.median(merged_df3[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df2[['NC3']].values / np.median(merged_df3[['NC3']].values))+ 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df2['High_Pause_Countsnc1'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
data.pause = paired_ratio
data.seq = sequence_embedding
# labeled_idx = torch.where(y.squeeze() != 0)[0]
# unlabeled_idx = torch.where(y.squeeze() == 0)[0]
# 假设merged_df2和merged_df3都有'transcript_id'这一列
# 建一个从transcript_id到索引的映射
# 先建一个 merged_df2['Unnamed: 0'] 到 index 的映射
id_to_idx = {tid: idx for idx, tid in enumerate(merged_df2['Unnamed: 0'])}

# 然后只查 intersection 中的
labeled_idx = [id_to_idx[tid] for tid in intersection]
# print(intersection)
# print(merged_df2['Unnamed: 0'].tolist()[:10])

# 转成 Tensor
labeled_idx = torch.tensor(labeled_idx, dtype=torch.long)

#print(f"最终labeled蛋白数量: {len(labeled_idx)}")  # 应该是4258


# 划分
# 划分

seed_everything(SEED)

train_idx, temp_idx = train_test_split(labeled_idx.cpu().numpy(), test_size=0.25, random_state=SEED)
test_idx, val_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED)

train_idx = torch.tensor(train_idx, device=device)
test_idx = torch.tensor(test_idx, device=device)
val_idx = torch.tensor(val_idx, device=device)

# pool里面是其他未标记的
# 假设你的样本总数是N（也就是y的样本数）
is_labeled = torch.zeros(y.size(0), dtype=torch.bool)
is_labeled[labeled_idx] = True  # 有标签的设为True
unlabeled_idx = torch.where(~is_labeled)[0]
pool_idx = unlabeled_idx.to(device)


# 初始化模型
#optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
# criterion = nn.MSELoss()
data = data.to(device)
data.x = data.x.to(device)
data.seq = data.seq.to(device)
data.pause = data.pause.to(device)
data.edge_index = data.edge_index.to(device)
y = y.to(device)

# 创建模型前，再设置一次种子 —— 确保模型的参数初始化是一样的
seed_everything(SEED)                 # ✅ 这里才是最关键的！
neural_net = NeuralGraph().to(device)

# 初始化优化器
optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
criterion = nn.MSELoss()

# 启动 self-learning
model = self_learning_process(
    data=data,
    y=y,
    model=neural_net,
    device=device,
    initial_labeled_idx=train_idx,
    val_idx=val_idx,
    pool_idx=pool_idx,
    batch_size=300,
    max_rounds=100
)

In [ ]:
# 1. 导入包
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import random
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score

# 2. 设置随机种子（必须在其他操作之前调用）
SEED = 8
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 在这里调用 `seed_everything(SEED)`
seed_everything(SEED)  # 固定整个环境的随机性

# 3. 早停器定义
class EarlyStopping:
    def __init__(self, patience=50):
        self.patience = patience
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def step(self, loss):
        if loss < self.best_loss:
            self.best_loss = loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

# 4. 训练函数
def train_model(model, data, train_idx, y_true, optimizer, criterion, patience=50):
    early_stopping = EarlyStopping(patience=patience)
    model.train()
    for epoch in range(10000):  # 大循环，由early stopping控制
        optimizer.zero_grad()
        out, _ = model(data)
        loss = criterion(out[train_idx], y_true[train_idx])
        loss.backward()
        optimizer.step()

        early_stopping.step(loss.item())
        if early_stopping.early_stop:
            print(f"Early Stopping at Epoch {epoch} with Loss {loss.item():.4f}")
            break

# 5. 验证函数
def evaluate_model(model, data, idx, y_true):
    model.eval()  # 确保在评估模式下
    with torch.no_grad():
        out, _ = model(data)
        loss = nn.MSELoss()(out[idx], y_true[idx])
        r2 = r2_score(y_true[idx].cpu().numpy(), out[idx].cpu().numpy())
    return loss.item(), r2

# 6. 主Self-Learning流程
def self_learning_process(data, y, model, device, initial_labeled_idx, val_idx, pool_idx, batch_size=300, max_rounds=10):
    optimizer = optim.Adam(model.parameters(), lr=7e-2)
    criterion = nn.MSELoss()

    # 7. 初始训练（确保模型在训练前种子已设置好）
    train_idx = initial_labeled_idx.clone()
    print(f"初始训练集大小: {len(train_idx)}")
    train_model(model, data, train_idx, y, optimizer, criterion)

    # 8. 伪标签循环（保证每轮开始前初始化随机数种子）
    for round_num in range(max_rounds):
        if len(pool_idx) == 0:
            print("没有更多未标记样本，结束Self-Learning。")
            break

        model.eval()
        with torch.no_grad():
            outputs, _ = model(data)
        
        # 从pool里选前batch_size个
        select_size = min(batch_size, len(pool_idx))
        selected_idx = pool_idx[:select_size]
        print(selected_idx)
        # 拿当前模型预测这些节点作为伪标签
        pseudo_labels = outputs[selected_idx].detach()

        # 更新y，把伪标签赋值
        y[selected_idx] = pseudo_labels

        # 将伪标签样本加入train_idx
        train_idx = torch.cat([train_idx, selected_idx], dim=0)

        # pool里移除这些样本
        pool_idx = pool_idx[select_size:]

        print(f"第{round_num+1}轮: 添加{select_size}个伪标签样本，总训练集大小{len(train_idx)}")

        # 9. 重新训练（确保每轮训练前随机种子初始化）
        optimizer = optim.Adam(model.parameters(), lr=7e-2)  # 每轮重新定义optimizer
        train_model(model, data, train_idx, y, optimizer, criterion)

    # 10. 最后在验证集上测试
    val_loss, val_r2 = evaluate_model(model, data, val_idx, y)
    print(f"最终验证集Loss: {val_loss:.4f}, 验证集R²: {val_r2:.4f}")
    return model

# 11. 数据准备
# 在这里确保数据准备（如 `labeled_idx` 的生成）是完全固定的
# 需要调用 `seed_everything(SEED)` 来保证数据划分的随机性一致

# 12. 初始化模型之前
neural_net = NeuralGraph().to(device)  # 神经网络模型初始化必须在 `seed_everything(SEED)` 后

# 13. 划分训练集、验证集，确保使用固定随机种子
train_idx, temp_idx = train_test_split(labeled_idx.cpu().numpy(), test_size=0.25, random_state=SEED)
test_idx, val_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED)

train_idx = torch.tensor(train_idx, device=device)
test_idx = torch.tensor(test_idx, device=device)
val_idx = torch.tensor(val_idx, device=device)

# 14. 定义未标记样本池
is_labeled = torch.zeros(y.size(0), dtype=torch.bool)
is_labeled[labeled_idx] = True  # 有标签的设为True
unlabeled_idx = torch.where(~is_labeled)[0]
pool_idx = unlabeled_idx.to(device)

# 15. 启动 self-learning 过程
model = self_learning_process(
    data=data,
    y=y,
    model=neural_net,
    device=device,
    initial_labeled_idx=train_idx,
    val_idx=val_idx,
    pool_idx=pool_idx,
    batch_size=300,
    max_rounds=100  # 最多做100轮
)


In [ ]:
import random
import numpy as np
import torch

SEED = 0  # 你自己定义的种子数，比如42或者任何数

# Python内置随机
random.seed(SEED)
# numpy随机
np.random.seed(SEED)
# torch随机
torch.manual_seed(SEED)
# 如果用的是GPU，也加上下面这行
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  # 如果有多个GPU

# 确保CUDA中的卷积算法确定性
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# 只选有标签的数据
X_cpm_log2 = np.log2((merged_df2[['rNC2']].values / np.median(merged_df3[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df2[['NC3']].values / np.median(merged_df3[['NC3']].values))+ 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df2['High_Pause_Countsnc1'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
data.pause = paired_ratio
data.seq = sequence_embedding
# labeled_idx = torch.where(y.squeeze() != 0)[0]
# unlabeled_idx = torch.where(y.squeeze() == 0)[0]
# 假设merged_df2和merged_df3都有'transcript_id'这一列
# 建一个从transcript_id到索引的映射
# 先建一个 merged_df2['Unnamed: 0'] 到 index 的映射
id_to_idx = {tid: idx for idx, tid in enumerate(merged_df2['Unnamed: 0'])}

# 然后只查 intersection 中的
labeled_idx = [id_to_idx[tid] for tid in intersection]

# 转成 Tensor
labeled_idx = torch.tensor(labeled_idx, dtype=torch.long)

print(f"最终labeled蛋白数量: {len(labeled_idx)}")  # 应该是4258


# 划分
# 划分
train_idx, temp_idx = train_test_split(labeled_idx.cpu().numpy(), test_size=0.25, random_state=SEED)
test_idx, val_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED)

train_idx = torch.tensor(train_idx, device=device)
test_idx = torch.tensor(test_idx, device=device)
val_idx = torch.tensor(val_idx, device=device)

# pool里面是其他未标记的
unlabeled_idx = torch.where(~is_labeled)[0]
# pool_idx = unlabeled_idx.to(device)
pool_idx = unlabeled_idx.cpu().numpy()
pool_idx = np.sort(pool_idx)  # 排序保证顺序一致
pool_idx = torch.tensor(pool_idx, device=device)


# 初始化模型
#optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
criterion = nn.MSELoss()
data = data.to(device)
data.x = data.x.to(device)
data.seq = data.seq.to(device)
data.pause = data.pause.to(device)
data.edge_index = data.edge_index.to(device)
y = y.to(device)
set_seed(SEED)
neural_net = NeuralGraph().to(device)

optimizer = optim.Adam(model.parameters(), lr=7e-2)  # 每轮重新定义optimizer

# 启动self-learning
model = self_learning_process(
    data=data,
    y=y,
    model=neural_net,
    device=device,
    initial_labeled_idx=train_idx,
    val_idx=val_idx,
    pool_idx=pool_idx,
    batch_size=300,
    max_rounds=100  # 最多做10轮
)


In [ ]:
import random
import numpy as np
import torch

SEED = 42  # 你自己定义的种子数，比如42或者任何数

# Python内置随机
random.seed(SEED)
# numpy随机
np.random.seed(SEED)
# torch随机
torch.manual_seed(SEED)
# 如果用的是GPU，也加上下面这行
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  # 如果有多个GPU

# 确保CUDA中的卷积算法确定性
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# 只选有标签的数据
X_cpm_log2 = np.log2((merged_df2[['rNC2']].values / np.median(merged_df3[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df2[['NC3']].values / np.median(merged_df3[['NC3']].values))+ 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df2['High_Pause_Countsnc1'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
data.pause = paired_ratio
data.seq = sequence_embedding
# labeled_idx = torch.where(y.squeeze() != 0)[0]
# unlabeled_idx = torch.where(y.squeeze() == 0)[0]
# 假设merged_df2和merged_df3都有'transcript_id'这一列
# 建一个从transcript_id到索引的映射
# 先建一个 merged_df2['Unnamed: 0'] 到 index 的映射
id_to_idx = {tid: idx for idx, tid in enumerate(merged_df2['Unnamed: 0'])}

# 然后只查 intersection 中的
labeled_idx = [id_to_idx[tid] for tid in intersection]

# 转成 Tensor
labeled_idx = torch.tensor(labeled_idx, dtype=torch.long)

print(f"最终labeled蛋白数量: {len(labeled_idx)}")  # 应该是4258


# 划分
# 划分
train_idx, temp_idx = train_test_split(labeled_idx.cpu().numpy(), test_size=0.25, random_state=SEED)
test_idx, val_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED)

train_idx = torch.tensor(train_idx, device=device)
test_idx = torch.tensor(test_idx, device=device)
val_idx = torch.tensor(val_idx, device=device)

# pool里面是其他未标记的
unlabeled_idx = torch.where(~is_labeled)[0]
pool_idx = unlabeled_idx.to(device)

# 初始化模型
#optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
criterion = nn.MSELoss()
data = data.to(device)
data.x = data.x.to(device)
data.seq = data.seq.to(device)
data.pause = data.pause.to(device)
data.edge_index = data.edge_index.to(device)
y = y.to(device)
set_seed(SEED)
neural_net = NeuralGraph().to(device)
# 启动self-learning
model = self_learning_process(
    data=data,
    y=y,
    model=neural_net,
    device=device,
    initial_labeled_idx=train_idx,
    val_idx=val_idx,
    pool_idx=pool_idx,
    batch_size=300,
    max_rounds=100  # 最多做10轮
)


In [ ]:
torch.save(model.state_dict(), './models/bulk_self_learning_best_state.pt')
print("模型已保存到 final_model.pth")
torch.save(model, './models/bulk_self_learning_best_full.pt')  # 保存整个对象

# # 加载
# model = torch.load('./modelfinal_full_model.pth')
# model = model.to(device)  # 如果需要的话，搬到cuda
# model.eval()


In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色
scatter_color = '#C7B0C4'
hist_color = '#BFD2DF'

# ✅ 只取4258个有标签的蛋白
# 假设你的全体预测是 y_pred，真实是 y_real，labeled_idx 是4258的索引
# 修改1️⃣：只取 labeled_idx 部分
original_y_np = y[labeled_idx].cpu().numpy().flatten()      # 真实值
y_all_pred_np = model(data)[0][labeled_idx].detach().cpu().numpy().flatten()  # 预测值

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np, y_all_pred_np)[0, 1]
result = pg.corr(original_y_np, y_all_pred_np, method='pearson')
p_value = result['p-val'][0]

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"
else:
    p_value_sci = f"{p_value:.2e}"
    p_base, p_exp = p_value_sci.split("e")
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)

# ➕ 对角线参考线
max_val = max(original_y_np.max(), y_all_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 🔁 回归线
sns.regplot(x=original_y_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression', fontsize=12)
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

ax_histy.yaxis.set_visible(False)
ax_histx.set_ylabel("Frequency")
ax_histx.xaxis.set_visible(False)

# 💾 保存
plt.savefig("./outputs/figures/bulk_self_learning_prediction_vs_observed.pdf", format="pdf", bbox_inches="tight")

# 📈 展示
plt.show()


In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色
scatter_color = '#C7B0C4'
hist_color = '#BFD2DF'

# ✅ 只拿 val_idx 的预测和真实
original_y_np = y[val_idx].cpu().numpy()
model.eval()
with torch.no_grad():
    y_all_pred, _ = model(data)

y_all_pred_np = y_all_pred[val_idx].cpu().numpy()

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np.flatten(), y_all_pred_np.flatten())[0, 1]
result = pg.corr(original_y_np.flatten(), y_all_pred_np.flatten(), method='pearson')
p_value = result['p-val'][0]

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"
else:
    p_value_sci = f"{p_value:.2e}"
    p_base, p_exp = p_value_sci.split("e")
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)

# ➕ 对角线参考线
max_val = max(original_y_np.max(), y_all_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 🔁 回归线
sns.regplot(x=original_y_np.flatten(), y=y_all_pred_np.flatten(), scatter=False, color='black', ax=ax_scatter)

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression (val set)', fontsize=12)
ax_scatter.set_ylabel('Predicted protein expression (val set)', fontsize=12)
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

ax_histy.yaxis.set_visible(False)
ax_histx.set_ylabel("Frequency")
ax_histx.xaxis.set_visible(False)

# 💾 保存
plt.savefig("./outputs/figures/bulk_self_learning_validation.pdf", format="pdf", bbox_inches="tight")

# 📈 展示
plt.show()


In [ ]:
# 只选有标签的数据
X_cpm_log2 = np.log2((merged_df2[['rNC2']].values / np.median(merged_df3[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df2[['NC3']].values / np.median(merged_df3[['NC3']].values))+ 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df2['High_Pause_Countsnc1'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
data.pause = paired_ratio
data.seq = sequence_embedding
labeled_idx = torch.where(y.squeeze() != 0)[0]
unlabeled_idx = torch.where(y.squeeze() == 0)[0]

# 划分
# 划分
train_idx, temp_idx = train_test_split(labeled_idx.cpu().numpy(), test_size=0.25, random_state=SEED)
test_idx, val_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED)

train_idx = torch.tensor(train_idx, device=device)
test_idx = torch.tensor(test_idx, device=device)
val_idx = torch.tensor(val_idx, device=device)

# pool里面是其他未标记的
unlabeled_idx = torch.where(~is_labeled)[0]
pool_idx = unlabeled_idx.to(device)

# 初始化模型
#optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
criterion = nn.MSELoss()
data = data.to(device)
data.x = data.x.to(device)
data.seq = data.seq.to(device)
data.pause = data.pause.to(device)
data.edge_index = data.edge_index.to(device)
y = y.to(device)
neural_net = NeuralGraph().to(device)
# 启动self-learning
model = self_learning_process(
    data=data,
    y=y,
    model=neural_net,
    device=device,
    initial_labeled_idx=train_idx,
    val_idx=val_idx,
    pool_idx=pool_idx,
    batch_size=300,
    max_rounds=100  # 最多做10轮
)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import random
from sklearn.metrics import r2_score

# ========== 设置随机种子 ==========
def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# ========== 模型定义 ==========
class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.regressor = nn.Linear(32, 1)
        self.regressor_activation = nn.Sequential(
            nn.ReLU()
        )

    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause

        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore)), dim=1)
        x = self.fc(x)

        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)

        out = self.regressor(z)
        out = self.regressor_activation(out)
        return out, z

# ========== Self-Learning函数 ==========
def self_learning(neural_net, data, X_train_idx, y_train, X_val_idx, y_val, X_test_idx, y_test, 
                  threshold=0.2, max_rounds=5):
    device = next(neural_net.parameters()).device
    optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
    criterion = nn.MSELoss()

    patience = 200
    num_epochs = 3000
    round_idx = 0

    for round_num in range(max_rounds):
        print(f"\n===== Self-learning Round {round_num+1}/{max_rounds} =====")

        best_val_loss = float('inf')
        patience_counter = 0
        best_model_wts = neural_net.state_dict()

        for epoch in range(1, num_epochs + 1):
            neural_net.train()
            optimizer.zero_grad()
            output, _ = neural_net(data)

            pred_train = output[X_train_idx]
            loss = criterion(pred_train, y_train.to(device))
            loss.backward()
            optimizer.step()

            neural_net.eval()
            with torch.no_grad():
                pred_val = output[X_val_idx]
                pred_test = output[X_test_idx]

                val_loss = criterion(pred_val, y_val.to(device)).item()
                val_r2 = r2_score(y_val.cpu().numpy(), pred_val.cpu().numpy())
                test_r2 = r2_score(y_test.cpu().numpy(), pred_test.cpu().numpy())

            if epoch % 50 == 0 or epoch == 1:
                print(f'Epoch {epoch:4d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f} | Val R²: {val_r2:.3f} | Test R²: {test_r2:.3f}')

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_val_r2 = val_r2
                best_test_r2 = test_r2
                patience_counter = 0
                best_model_wts = neural_net.state_dict()
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        # 加载最优模型
        neural_net.load_state_dict(best_model_wts)
        print(f"Round {round_num+1} Done! Best Val Loss: {best_val_loss:.4f} | Val R²: {best_val_r2:.3f} | Test R²: {best_test_r2:.3f}")

        # ========== Self-learning 选择高置信度伪标签样本 ==========
        neural_net.eval()
        with torch.no_grad():
            output, _ = neural_net(data)
            pred_test = output[X_test_idx]

            # 计算误差
            error = torch.abs(pred_test.squeeze() - y_test.to(device).squeeze())

            # 选取误差小于阈值的样本
            confident_idx = torch.where(error < threshold)[0]
            if len(confident_idx) == 0:
                print("没有新的高置信度样本，Self-learning终止。")
                break

            # 把这些样本加到训练集
            selected_idx = X_test_idx[confident_idx.cpu().numpy()]
            selected_labels = y_test[confident_idx.cpu().numpy()]

            print(f"Round {round_num+1}: 加入 {len(selected_idx)} 个伪标签样本到训练集中")

            # 更新训练集
            X_train_idx = torch.cat([X_train_idx, torch.tensor(selected_idx, device=device)], dim=0)
            y_train = torch.cat([y_train, selected_labels.to(device)], dim=0)

            # 从测试集中移除这些样本
            mask = torch.ones(len(X_test_idx), dtype=torch.bool, device=device)
            mask[confident_idx] = False
            X_test_idx = X_test_idx[mask]
            y_test = y_test[mask.cpu().numpy()]

        if len(X_test_idx) == 0:
            print("测试集已经空了，Self-learning终止。")
            break

    return neural_net

# ========== 开始训练 ==========
SEED = 12
set_seed(SEED)

device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
neural_net = NeuralGraph().to(device)
X_cpm_log2 = np.log2((merged_df2[['rNC2']].values / np.median(merged_df2[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df2[['NC3']].values / np.median(merged_df2[['NC3']].values))+ 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df2['High_Pause_Countsnc1'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
data.pause = paired_ratio
data.seq = sequence_embedding

# 你的Data对象
data = data.to(device)
X_train, X_temp, y_train, y_temp, train_idx_X, temp_idx_X = train_test_split(
    X, y, np.arange(len(X)), test_size=0.25, random_state=SEED)

X_val, X_test, y_val, y_test, val_idx_X, test_idx_X = train_test_split(
    X_temp, y_temp, temp_idx_X, test_size=1/3, random_state=SEED)

sequence_embedding_train, sequence_embedding_temp, train_idx_seq, temp_idx_seq = train_test_split(
    sequence_embedding, np.arange(len(sequence_embedding)), test_size=0.25, random_state=SEED)

sequence_embedding_val, sequence_embedding_test, val_idx_seq, test_idx_seq = train_test_split(
    sequence_embedding_temp, temp_idx_seq, test_size=1/3, random_state=SEED)

pause_train, pause_temp, train_idx_pause, temp_idx_pause = train_test_split(
    paired_ratio, np.arange(len(paired_ratio)), test_size=0.25, random_state=SEED)

pause_val, pause_test, val_idx_pause, test_idx_pause = train_test_split(
    pause_temp, temp_idx_pause, test_size=1/3, random_state=SEED)

train_idx_X = torch.tensor(train_idx_X, dtype=torch.long, device=device)
val_idx_X = torch.tensor(val_idx_X, dtype=torch.long, device=device)
test_idx_X = torch.tensor(test_idx_X, dtype=torch.long, device=device)
y_train = y_train.to(device)
y_val = y_val.to(device)
y_test = y_test.to(device)

# Self-learning
neural_net = self_learning(
    neural_net, data,
    train_idx_X=torch.tensor(train_idx_X, device=device),
    y_train=y_train,
    X_val_idx=torch.tensor(val_idx_X, device=device),
    y_val=y_val,
    X_test_idx=torch.tensor(test_idx_X, device=device),
    y_test=y_test,
    threshold=0.2,  # 控制伪标签的置信度
    max_rounds=5    # 最多做5轮self-learning
)

print("Self-learning 训练结束！🚀")


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.model_selection import train_test_split

# ========== 数据准备 ==========

# 1. ribo-seq & protein abundance log2(CPM)归一化
X_ribo = np.log2((merged_df2[['rNC2']].values / np.median(merged_df2[['rNC2']].values)) + 1)
y_protein = np.log2((merged_df2[['NC3']].values / np.median(merged_df2[['NC3']].values)) + 1)

# 2. 转成Tensor
X_ribo = torch.tensor(X_ribo, dtype=torch.float32)
y_protein = torch.tensor(y_protein, dtype=torch.float32)

# 3. 额外特征：pause、sequence
pause =  torch.tensor(np.array(merged_df2['High_Pause_Countsnc1'], dtype=np.float32).reshape(-1, 1))

sequence = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)

# 4. 拼接所有特征
X_all = torch.cat([X_ribo, pause, sequence], dim=1)
ppi_matrix = sp.coo_matrix(ppi_matrix)
# 5. 图结构
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 6. 只保留有蛋白质表达量的节点
labeled_mask = (y_protein.squeeze() != 0)

# 7. 拆分训练/验证（比如80% train，20% valid）
labeled_indices = labeled_mask.nonzero(as_tuple=True)[0]
train_idx, val_idx = train_test_split(labeled_indices.numpy(), test_size=0.2, random_state=42)

# ========== 构建图对象 ==========
data = Data(x=X_all, edge_index=edge_index, edge_attr=edge_weight, y=y_protein)


# ========== 定义GNN模型 ==========
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, x, edge_index, edge_weight):
        x = self.conv1(x, edge_index, edge_weight=edge_weight)
        x = torch.relu(x)
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        return x


# ========== 训练函数 ==========
def train(model, optimizer, data, train_idx):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.edge_attr)
    loss = nn.MSELoss()(out[train_idx], data.y[train_idx])
    loss.backward()
    optimizer.step()
    return loss.item()


def evaluate(model, data, val_idx):
    model.eval()
    out = model(data.x, data.edge_index, data.edge_attr)
    val_loss = nn.MSELoss()(out[val_idx], data.y[val_idx])
    return val_loss.item(), out


# ========== Self-Learning循环 ==========
input_dim = data.x.shape[1]
hidden_dim = 64
output_dim = 1

model = GCN(input_dim, hidden_dim, output_dim)
optimizer = optim.Adam(model.parameters(), lr=0.001)

all_unlabeled_idx = (~labeled_mask).nonzero(as_tuple=True)[0]

n_self_learning_rounds = 5
n_pseudo_labels_per_round = 300  # 每轮加300个伪标签

for round in range(n_self_learning_rounds):
    print(f'===== Round {round+1} / {n_self_learning_rounds} =====')
    
    # 1. 正常训练几轮
    for epoch in range(50):  # 每轮50个epoch
        loss = train(model, optimizer, data, torch.tensor(train_idx))
        if (epoch+1) % 10 == 0:
            val_loss, _ = evaluate(model, data, torch.tensor(val_idx))
            print(f'Epoch {epoch+1}, Train Loss: {loss:.4f}, Val Loss: {val_loss:.4f}')

    # 2. 用模型在未标记节点上预测
    model.eval()
    all_out = model(data.x, data.edge_index, data.edge_attr).detach()

    # 3. 挑出预测最自信的一批节点
    preds_unlabeled = all_out[all_unlabeled_idx]
    confidences = torch.abs(preds_unlabeled.squeeze())
    topk_idx = torch.topk(confidences, min(n_pseudo_labels_per_round, len(all_unlabeled_idx)), largest=True).indices

    selected_idx = all_unlabeled_idx[topk_idx]

    # 4. 把这些节点加到训练集中，伪标签直接用模型预测值
    pseudo_labels = preds_unlabeled[topk_idx].detach()
    data.y[selected_idx] = pseudo_labels

    train_idx = np.concatenate([train_idx, selected_idx.numpy()])

    # 5. 从unlabeled pool中去掉这些节点
    all_unlabeled_idx = torch.tensor([idx for idx in all_unlabeled_idx if idx not in selected_idx])

print("\nTraining Finished!")


In [ ]:
# 1. 区分有标签蛋白和无标签蛋白
labeled_mask = merged_df2['NC3'] != 0  # 有蛋白质表达量的
unlabeled_mask = merged_df2['NC3'] == 0  # 没蛋白质表达量的

labeled_indices = np.where(labeled_mask)[0]
unlabeled_indices = np.where(unlabeled_mask)[0]

print(f"有标签蛋白数量: {len(labeled_indices)}")
print(f"无标签蛋白数量: {len(unlabeled_indices)}")

# 2. 划分有标签蛋白为训练集、验证集、测试集
from sklearn.model_selection import train_test_split

train_idx, temp_idx = train_test_split(labeled_indices, test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

print(f"训练集: {len(train_idx)}, 验证集: {len(val_idx)}, 测试集: {len(test_idx)}")

# 3. 创建 Data 对象
# 假设你的X是riboseq表达，y是蛋白质表达量
X_labeled = X[labeled_indices]
y_labeled = y[labeled_indices]
pause_labeled = paired_ratio[labeled_indices]
seq_labeled = sequence_embedding[labeled_indices]

X_unlabeled = X[unlabeled_indices]
pause_unlabeled = paired_ratio[unlabeled_indices]
seq_unlabeled = sequence_embedding[unlabeled_indices]

# 有标签训练数据
train_data = Data(
    x=X_labeled[train_idx],
    edge_index=data.edge_index,
    edge_attr=data.edge_attr,
    y=y_labeled[train_idx]
)
train_data.pause = pause_labeled[train_idx]
train_data.seq = seq_labeled[train_idx]
train_data = train_data.to(device)

# 验证数据
val_data = Data(
    x=X_labeled[val_idx],
    edge_index=data.edge_index,
    edge_attr=data.edge_attr,
    y=y_labeled[val_idx]
)
val_data.pause = pause_labeled[val_idx]
val_data.seq = seq_labeled[val_idx]
val_data = val_data.to(device)

# 测试数据
test_data = Data(
    x=X_labeled[test_idx],
    edge_index=data.edge_index,
    edge_attr=data.edge_attr,
    y=y_labeled[test_idx]
)
test_data.pause = pause_labeled[test_idx]
test_data.seq = seq_labeled[test_idx]
test_data = test_data.to(device)

# 无标签数据
unlabeled_data = Data(
    x=X_unlabeled,
    edge_index=data.edge_index,
    edge_attr=data.edge_attr,
    y=torch.zeros(len(unlabeled_indices), 1)  # dummy
)
unlabeled_data.pause = pause_unlabeled
unlabeled_data.seq = seq_unlabeled
unlabeled_data = unlabeled_data.to(device)
